# ZeroPoint against hosted LLMs

Runs Anthropic and OpenAI models as decision engines and scores them next to ZeroPoint and Laya with the same
`report.py`. With the default settings it reproduces the comparison published in BENCHMARK.md.

How each model is asked is fixed in `benchmarks/llm/run_llm.py`: the text and the options plus *none of these*, one
tool call (Anthropic) or a strict JSON schema (OpenAI) with the choice as an enum and a stated confidence from 0 to 1.
No examples. The models read three tests only, the ones they are unlikely to have seen in training (the 2,000-case
test and the two support-ticket tests), on the same fixed 300-case sample of each as every other system: 900 calls
per model. Works on Google Colab.

In [ ]:
%cd /content
!git clone -q https://github.com/Javihaus/zeropoint.git
%cd /content/zeropoint
!pip install -q -e . anthropic openai datasets pandas scikit-learn matplotlib

## 1. Rebuild the tests

The test texts are not in the repository: this rebuilds them from pinned dataset versions and checks their hashes.

In [ ]:
!cd /content/zeropoint && python benchmarks/compare/build_tests.py
!cd /content/zeropoint/benchmarks/compare/tests && sha256sum -c SHA256SUMS

## 2. Keys, models and prices

API keys come from Colab secrets: add `ANTHROPIC_API_KEY` and `OPENAI_API_KEY` (the key icon on the left) and turn
on *Notebook access* for each. An Anthropic API key starts with `sk-ant-api`.

Prices are in USD per million tokens (input, output), from each provider's price page on the day you run this. They
only add the cost per decision; the published run did not record prices.

In [ ]:
import os
try:
    from google.colab import userdata
    for name in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY"):
        try:
            os.environ[name] = userdata.get(name).strip()
        except Exception:                      # secret not defined, or notebook access not granted
            pass
except ImportError:                            # not on Colab: keys from the environment
    pass
for name in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY"):
    key = os.environ.get(name, "")
    print(name, "→", key[:14] + "…" if key else "not set")

ANTHROPIC_MODELS = ["claude-opus-5-5", "claude-sonnet-5-5", "claude-haiku-5-5"]
OPENAI_MODELS = ["gpt-6-luna", "gpt-6.1-terra", "gpt-6.1-sol"]

PRICES = {                  # (input, output) USD per million tokens; None: no cost computed
    m: (None, None) for m in ANTHROPIC_MODELS + OPENAI_MODELS
}

In [ ]:
import re
ROOT = "/content/zeropoint" if os.path.isdir("/content/zeropoint") else os.getcwd()
os.chdir(ROOT)                              # every later cell runs from the repository's root
path = os.path.join(ROOT, "benchmarks/llm/run_llm.py")
src = open(path).read()
src = re.sub(r"PRICES = \{.*?\n\}", "PRICES = " + repr(PRICES).replace("), ", "),\n    "), src, flags=re.S)
open(path, "w").write(src)
print("prices set in", path)

## 3. Run the models

Every model at once, each in its own process, with the defaults of `run_llm.py`: 900 calls per model. Add
`"--shuffle"` to the command to also measure whether the order of the options changes the answer (900 more calls).
Answers are saved as they come: if the session stops, run this cell again and each model continues where it was.

In [ ]:
import subprocess
os.makedirs("logs", exist_ok=True)
jobs = [("anthropic", m) for m in ANTHROPIC_MODELS if os.environ.get("ANTHROPIC_API_KEY")] + \
       [("openai", m) for m in OPENAI_MODELS if os.environ.get("OPENAI_API_KEY")]
procs = {m: subprocess.Popen(["python", "benchmarks/llm/run_llm.py", p, m],
                             stdout=open(f"logs/{m}.log", "a"), stderr=subprocess.STDOUT)
         for p, m in jobs}
print("running:", list(procs))

In [ ]:
# Progress: run this cell whenever you like. When every model shows "done", go on to step 4.
for m, pr in procs.items():
    lines = open(f"logs/{m}.log").read().strip().splitlines()
    print(f"{m:20s} {'done' if pr.poll() is not None else 'running':8s} {lines[-1] if lines else '(starting)'}")

## 4. Score everything together

The same tables as in BENCHMARK.md, section *Compared with hosted LLMs*.

In [ ]:
!cd /content/zeropoint && python benchmarks/compare/report.py > /dev/null
from IPython.display import Markdown
text = open("/content/zeropoint/benchmarks/compare/results/tables.md").read()
Markdown(text[text.index("## Compared with hosted LLMs"):text.index("## The tests")])

## 5. Accuracy against latency and cost

From `summary.json`, the same numbers as the tables: the 300-case sample of the three tests, 900 cases per system.
ZeroPoint and Laya run locally, so their cost per decision is drawn at the left edge.

In [ ]:
import json, matplotlib.pyplot as plt
C = json.load(open("/content/zeropoint/benchmarks/compare/results/summary.json"))
names = {s: r["name"] for s, r in C["systems"].items()}
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for s, r in C["llm_comparison"]["systems"].items():
    cost = (C["systems"][s].get("cost_usd_per_decision") or 1e-7)
    for a, x in [(ax[0], r["ms_median"]), (ax[1], cost)]:
        a.scatter(x, 100 * r["right"], s=60)
        a.annotate(f"{names[s]}\nsilent mistakes {100 * r['silent']:.0f}%", (x, 100 * r["right"]),
                   textcoords="offset points", xytext=(6, 4), fontsize=8)
ax[0].set(xscale="log", xlabel="median latency per decision, ms (log)", ylabel="right decision, none included (%)")
ax[1].set(xscale="log", xlabel="cost per decision, USD (log; local models at the left edge)")
for a in ax: a.grid(alpha=.3)
fig.suptitle("Mean over " + ", ".join(C["llm_comparison"]["tests"]) + " (300-case sample each)", fontsize=10)
plt.tight_layout(); plt.savefig("/content/zeropoint/benchmarks/llm/accuracy_latency_cost.png", dpi=150)